In [ ]:
import threading
import time


# 创建线程方法一：函数方式
def task(name):
    print(f"{name} start")
    time.sleep(2)
    print(f"{name} end")

# 创建线程
t1 = threading.Thread(target=task,args=("线程1",))
t2 = threading.Thread(target=task,args=("线程2",))

# 启动线程
t1.start()
t2.start()

# 等待线程结束
t1.join() # join() 的作用 - 检测子线程是否结束 - 没结束，主线程等待，结束了，主线程继续往下走
t2.join()

print("All threads complete.")

In [ ]:
# 创建线程方式二：继承方式

class MyThread(threading.Thread):
    def __init__(self, name):
        super().__init__()
        self.name = name
    
    def run(self): # 必须重写run方法
        print(f"{self.name} 执行中")
        time.sleep(1)
        print(f"{self.name} 结束")

t1 = MyThread("线程1")
t2 = MyThread("线程2")

t1.start()
t2.start()

t1.join()
t2.join()

print("All threads complete.")

In [ ]:
# 线程锁 - Lock

# 没有锁：结果可能混乱
count = 0
def add_without_lock():
    global count
    for _ in range(1000000):
        count +=1 

t1 = threading.Thread(target=add_without_lock)
t2 = threading.Thread(target=add_without_lock)

t1.start()
t2.start()
t1.join()
t2.join()
print(count) # 想演示的结果是：期望是 200000，实际可能是 153482 等
"""
比如：
count = 5

线程1 count = 5
线程2 count = 5 - 此时线程1 还没写回，线程2 几乎同时读到了5

线程1 - 5 + 1 = 6 ， 写回6
线程2 - 5 + 1 = 6 , 写回6 - 但其实这里理想情况下应该是 6 + 1 = 7 写回7

这就导致了，两个线程都加了一次，但是count结果值只加了1次，从结果上来看丢了一次。

- 无法复现原因：数字太小 or python GIL锁限制
"""

In [ ]:
# 有锁，结果一定正确
lock = threading.Lock()
count = 0
def add_with_lock():
    global count
    for _ in range(10000):
        with lock: # 利用上下文管理器管理锁，用完自动释放
            count +=1

t1 = threading.Thread(target=add_with_lock)
t2 = threading.Thread(target=add_with_lock)

t1.start()
t2.start()
t1.join()
t2.join()
print(count)


"""
count 当前值 = 5

线程1：加锁，读取 count = 5，+1 = 6，写回并释放锁
线程2：发现锁被占用，等待...
线程2：线程1释放锁，加锁，读取 count = 6，+1 = 7，写回bing释放锁

结果正确，count = 7
"""

In [ ]:
# 接口自动化测试比较常用并发请求

def send_request(url, results, index):
    print(f"请求 {url} 开始")
    time.sleep(1) # 模拟网络请求耗时
    results[index] = f"{url} 响应成功"
    print(f"请求 {url} 完成")

urls = ["/api/login", "/api/user", "/api/order"]
results = [None] * len(urls)
threads = []

# 同时发起所有请求
for i, url in enumerate(urls):
    t = threading.Thread(target=send_request, args=(url, results, i))
    threads.append(t)
    t.start()

# 等待所有请求完成
for t in threads:
    t.join()

print(f"所有结果: {results}")

In [ ]:
# 线程池

# 用法一：submit()
from concurrent.futures import ThreadPoolExecutor
import time

def task(name):
    print(f"{name} start")
    time.sleep(1)
    print(f"{name} end")

# max_worker 控制线程池大小
with ThreadPoolExecutor(max_workers=3) as pool:
    future1 = pool.submit(task, "Thread1")
    future2 = pool.submit(task, "Thread2")
    future3 = pool.submit(task, "Thread3")

    # 获取返回值 - 会等待任务完成
    print(future1.result()) 
    print(future2.result()) 
    print(future3.result()) 

"""
submit() 返回的是Future对象
Future对象可以查询任务状态 - done() False 还没完成，True已经完成。 running() - True 正在执行 False 已结束
"""


In [ ]:
# 用法二：map() - 适合同一个函数处理多个参数 比如接口自动化

def send_request(url):
    time.sleep(1)
    return f"{url} 响应成功"

urls = ["/api/login", "/api/user", "/api/order", "/api/goods", "/api/pay"]

with ThreadPoolExecutor(max_workers=3) as pool:
    # 自动把urls里的每个元素传给send)request
    results = pool.map(send_request, urls)

    for res in results:
        print(res)

总结：
1. 多线程是让程序同时执行多个任务，python是用threading模块来实现。
2. join() 的所用是等待子线程结束，如果没有结束继续等待，结束了主线程会继续往下走。 不加join() 可能会出现，子线程还在跑，但主线程自顾自结束了的情况。
3. python 有GIL机制，全局解释锁，同一时间只允许一个线程执行python代码。所以python不支持CPU密集型（用多进程实现），但是支持IO密集型（网络请求，文件读写）。
4.多线程适合接口自动化，可以同时对多个接口发送请求。
5.多个进程操作同一个共享变量时，记得要加Lock。可以用with 管理，避免忘记关掉锁而导致程序锁死。
6.线程池可以提前创建好一批线程，且线程可以复用，不用每次任务都创建新线程造成额外开销。
7.submit() 逐个提交，可以获取每个Future对象 ，map() 可以处理多个参数，批量提交按顺序返回结果。 但是map()是按顺序返回结果给results，如果task1 一直不结束就会一直等待，as_complete 则是先到先的。
  如果不需要按顺序收集结果，则submit() + as_complete 更灵活一点。

本质上就是进程和线程的区别：

CPU 密集型（大量计算）：
线程1 一直在算，一直占着 GIL 不放
→ 线程2 只能干等
→ 两个线程实际是排队执行，和单线程一样慢
→ 甚至因为线程切换的开销，比单线程还慢！

I/O 密集型（网络请求、文件读写）：
线程1 发出网络请求，开始等待响应
→ 等待期间 GIL 自动释放！
→ 线程2 趁机拿到 GIL 开始执行
→ 两个线程真正交替利用了等待时间
→ 效率大幅提升！

进程：
每个进程有自己独立的内存空间和 GIL
→ 真正的并行，互不影响
→ 开销大（每个进程都要分配独立内存）

线程：
同一个进程下的多个线程共享内存和 GIL
→ 受 GIL 限制，不能真正并行执行 Python 代码
→ 开销小（共享内存，创建快）

今天这个实战题有些难度，涉及知识点比较广泛，但是有些坎我们总得迈过去，对吗？

题目:高效文件下载管理器

1. 实现一个高效的多线程文件下载管理器，用户可以输入一个网址列表，程序会自动使用多线程下载这些网址对应的文件。
2. 使用Python的 concurrent.futures.ThreadPoolExecutor 来创建线程池，并允许用户设置最大线程数。
3. 对于每个下载任务，实时显示下载进度条，并在下载完成后显示总下载速度和下载时间
4. 下载完成后，输出每个文件的大小、下载时间和平均下载速度。

提示：
1. 使用 requests 库来下载文件，可以使用 stream=True 参数来实现分块下载。
2. 使用 os 和 shuti1 库来处理文件，例如创建目录、移动文件等。
3. 使用 time 和 datetime 库来计算下载速度和下载时间,
4. 使用 tqdm 库来显示进度条。

更详细的提示：
1. 使用requests库获取文件大小：
   在发送请求时，使用stream=True参数，这样requests不会立即下载整个文件，而是允许您流式地处理它。
   通过response.headers可以获取到Content-Length，这是服务器返回的文件大小。
2. 使用concurrent.futures.ThreadPoolExecutor创建线程池：
   创建一个ThreadPoolExecutor实例，并设置最大线程数。
   使用executor.submit()方法提交下载任务到线程池。
3. 显示下载进度条：
   使用tqdm库创建一个进度条对象，设置total参数为文件的总大小。
   在下载文件的过程中，不断更新进度条对象的update()方法，以显示进度。
4. 计算下载速度和下载时间：
   在开始下载前记录时间，下载完成后再次记录时间，计算时间差即为下载时间。
   下载速度可以通过总下载大小除以下载时间来计算。
5. 处理文件：
   使用os库来创建目录，确保保存文件的目录存在。
   使用shutil库来移动文件，如果需要的话。
6. 收集和输出下载信息：
   为每个下载任务创建一个字典，存储文件大小、下载时间和下载速度等信息。
   在所有任务完成后，输出这些信息。

实现：
一、先别着急写代码，第一步先拆解需求：

根据题目，其实要求要实现5个功能：
1. 下载文件
2. 显示进度条
3. 统计速度
4. 汇总结果

-> 所以拆成：先创建 download_file() 方法

实现：
下载文件；显示进度；统计速度；返回结果

然后 main() 负责：
创建线程池
提交多个下载任务
收集结果
打印结果

url = "https://httpbin.org/get"

url = "https://httpbin.org/bytes/10240"


In [ ]:
# 第一步：实现下载一个文件的代码代码块
import requests

url = "https://httpbin.org/bytes/10240"

response = requests.get(url,stream=True) # stream = True 流式处理，边下载边读取
print(response.status_code)

In [ ]:
# 第二步：获取文件大小
total_size = int(
    response.headers.get("Content-Length",0)
)
print(total_size)


In [ ]:
# 第三步：保存文件
with open("test.zip","wb") as f:
    for chunk in response.iter_content( # 边下载边写入文件 以chunk为字节单位写入
        chunk_size=1024
    ):
        f.write(chunk)

In [ ]:
# 第四步：统计时间
import time

start_time = time.time()

end_time = time.time()

download_time = end_time - start_time

In [ ]:
# 第五步：统计速度
speed = total_size / download_time

In [ ]:
# 第六步：创建进度条
from tqdm import tqdm

# 创建进度条
progress = tqdm(
    total=total_size,
    unit = "B", # 进度条单位：Byte，告诉tqdm当前统计的是字节数 如果unit = "file" 则告诉tqdm当前统计的是file数
    unit_scale=True # 自动换算单位，不用全量展开字节数和文件大小
)

# 每下载一块就更新一下进度条
for chunk in response.iter_content(1024):
    f.write(chunk)
    progress.update(len(chunk))

In [ ]:
# 封装成函数
import requests
import time
from tqdm import tqdm
import os

def download_file(url):
    start_time = time.time()

    headers = {
    "User-Agent":
    "Mozilla/5.0"
} # 绕过被测网站对python 的防范
    
    response = requests.get(url,stream=True,verify=False,headers=headers)
    # 如果end_time 放在这里，统计的只是服务器返回响应头的时间而不是下载文件的时间

    if response.status_code !=200:
        raise Exception(
            f"Download Failed: {response.status_code} - {url}"
        )

    total_size = int(response.headers.get("Content-Length",0)) # 记得转换数据类型

    progress = tqdm(
        total=total_size,
        unit="B",
        unit_scale=True

    )
    
    # 边下载边写入，并更新进度条
    # 先创建download file的文件夹，方便管理
    download_dir = "downloads"

    if not os.path.exists(download_dir):
        os.makedirs(download_dir)

    filename = url.split("/")[-1]

    filepath = os.path.join(
        download_dir,
        filename
    )

    with open(filepath,"wb") as f:
        for chunk in response.iter_content(
            chunk_size=1024
        ):
            f.write(chunk)
            progress.update(len(chunk)) # update就是终端操作，无需在retrun输出
    
    progress.close() # 不关闭进度条，多线程时终端显示会很混乱

    end_time = time.time()
    download_time = end_time - start_time
    speed = total_size / download_time

    return {
        "url":url,
        "size":total_size,
        "time":round(download_time,2),
        "speed":round(speed,2)
    }

    

In [ ]:
download_file("https://res.wx.qq.com/open/libs/weui/2.5.16/weui.min.css")

In [3]:
download_file("https://httpbin.org/bytes/10240")

TypeError: timer.<locals>.wrapper() takes 0 positional arguments but 1 was given

In [83]:
from concurrent.futures import ThreadPoolExecutor , as_completed

urls = [
    "https://httpbin.org/bytes/10240",
    "https://httpbin.org/bytes/20480",
    "https://httpbin.org/bytes/40960",
    "https://ipv4.download.thinkbroadband.com/5MB.zip"
]

results = []

with ThreadPoolExecutor(max_workers=3) as pool:

    future_dict = {
        pool.submit(download_file, url=url): url
        for url in urls
    }

    for future in as_completed(future_dict):

        url = future_dict[future]

        try:
            result = future.result()

            results.append(result)

            print(f"[Complete] {url}")

        except Exception as e:

            print(f"[Failed] {url}")
            print(e)
    
    for res in results:
        print(f"Downloaded files' details :{res}")



/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '127.0.0.1'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '127.0.0.1'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '127.0.0.1'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.

[Complete] https://httpbin.org/bytes/10240


100%|██████████| 20.5k/20.5k [00:00<00:00, 220kB/s]


[Complete] https://httpbin.org/bytes/20480


100%|██████████| 41.0k/41.0k [00:00<00:00, 147kB/s] 


[Complete] https://httpbin.org/bytes/40960


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '127.0.0.1'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


[Failed] https://ipv4.download.thinkbroadband.com/5MB.zip
Download Failed: 403 - https://ipv4.download.thinkbroadband.com/5MB.zip
Downloaded files' details :{'url': 'https://httpbin.org/bytes/10240', 'size': 10240, 'time': 2.43, 'speed': 4217.98}
Downloaded files' details :{'url': 'https://httpbin.org/bytes/20480', 'size': 20480, 'time': 2.52, 'speed': 8130.79}
Downloaded files' details :{'url': 'https://httpbin.org/bytes/40960', 'size': 40960, 'time': 2.74, 'speed': 14935.25}


In [ ]:
from tqdm import tqdm
import time

p = tqdm(total=100)

for i in range(100):
    time.sleep(0.1)
    p.update(1)

In [ ]:
import requests

response = requests.get(
    "https://httpbin.org/bytes/10240",
    stream=True,
    verify=False
)

print(response.status_code)

In [ ]:
import os

print(os.environ.get("HTTP_PROXY"))
print(os.environ.get("HTTPS_PROXY"))

总结：
1. 先拆解需求，题目首先要实现5个功能 -> 下载文件、显示进度条、统计速度、汇总结果 -> 这些是download_file() 方法中需要处理的逻辑。
   然后在main()函数中，创建线程池调用download_file() 启动多个下载任务，收集并打印结果。

2. debug download_file() 时，写入文件写的是-> with open(text.txt,"wb") as f -> 在debug 线程池操作时要记得改成filename，支持多文件，不然会写入冲突。

3. end_time 的时机应该刚在写入文件后。 因为response只是将内容存在了content中，还没有全部下载（stream = True)。end_time 紧跟response之后统计的则只是得到响应的时间。
4. progress.close() 要及时记得关闭进度条，避免console输出混乱。

5. 多下载文件任务，使用future = submit() + as_completed(future) “先到先得”，那个并发任务先完成可以先输出结果，不用非要按顺序等待。